# Train 4-class classifier from `input_test` images

This notebook is designed for **Google Colab**. It trains a 4-class CNN from MATLAB-generated `input_test` images and `output_test` labels.

## Why 4 classes?
The labels are ordinal severity bins derived from cortical thickness reduction relative to a reference thickness of `7e-3 m`:

- **Class 1**: `<10%` → Normal
- **Class 2**: `10–20%` → Osteopenia
- **Class 3**: `20–30%` → Osteoporosis
- **Class 4**: `>30%` → Severe osteoporosis

This discretization is useful because it turns a continuous thickness-loss variable into interpretable severity levels and makes the proof-of-concept task a classification problem rather than exact regression.

In [ ]:
# If needed on Colab, uncomment:
# !pip install h5py scikit-learn torch torchvision --quiet

In [2]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [1]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

torch version: 2.10.0+cu128
cuda available: True


In [ ]:
# =========================
# Configuration
# =========================
MAT_PATH = '/content/Test_data_sobol_labels_layer1.mat'
OUTPUT_DIR = Path('/content/training_runs/cnn_4class')
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 42

CLASS_INFO = {
    0: {'name': 'Normal', 'range': '<10%'},
    1: {'name': 'Osteopenia', 'range': '10-20%'},
    2: {'name': 'Osteoporosis', 'range': '20-30%'},
    3: {'name': 'Severe osteoporosis', 'range': '>30%'},
}

## Upload the `.mat` file
Upload `Test_data_sobol_labels_layer1.mat` to Colab before running the next cells.

In [ ]:
# Optional upload helper in Colab
from google.colab import files
uploaded = files.upload()

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_mat_dataset(mat_path: str | Path):
    with h5py.File(mat_path, 'r') as f:
        images = np.array(f['input_test'])
        labels = np.array(f['output_test'])

    if images.ndim != 3:
        raise ValueError(f'Expected 3D tensor, got {images.shape}')

    if images.shape[0] != labels.shape[0] and images.shape[-1] == labels.shape[0]:
        images = np.transpose(images, (2, 1, 0))
    elif images.shape[0] == labels.shape[0]:
        pass
    else:
        raise ValueError(f'Cannot align images {images.shape} with labels {labels.shape}')

    labels = labels.reshape(-1).astype(np.int64)
    labels = labels - 1  # MATLAB 1..4 -> Python 0..3
    images = images.astype(np.float32)
    return images, labels


class DispersionImageDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray):
        self.images = images
        self.labels = labels

    def __len__(self):
        return int(self.labels.shape[0])

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index]).unsqueeze(0)
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return image, label


class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


@dataclass
class SplitData:
    train_images: np.ndarray
    val_images: np.ndarray
    test_images: np.ndarray
    train_labels: np.ndarray
    val_labels: np.ndarray
    test_labels: np.ndarray


def stratified_split(images, labels, seed):
    train_images, test_images, train_labels, test_labels = train_test_split(
        images, labels, test_size=0.2, random_state=seed, stratify=labels
    )
    train_images, val_images, train_labels, val_labels = train_test_split(
        train_images, train_labels, test_size=0.2, random_state=seed, stratify=train_labels
    )
    return SplitData(train_images, val_images, test_images, train_labels, val_labels, test_labels)


def build_loaders(split: SplitData, batch_size: int):
    train_loader = DataLoader(DispersionImageDataset(split.train_images, split.train_labels), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(DispersionImageDataset(split.val_images, split.val_labels), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(DispersionImageDataset(split.test_images, split.test_labels), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def run_epoch(model, loader, criterion, optimizer, device):
    is_train = optimizer is not None
    model.train(is_train)
    losses, all_targets, all_preds = [], [], []

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, targets)
            if is_train:
                loss.backward()
                optimizer.step()

        losses.append(float(loss.detach().cpu().item()))
        preds = torch.argmax(logits, dim=1)
        all_targets.append(targets.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())

    return float(np.mean(losses)), np.concatenate(all_targets), np.concatenate(all_preds)


def evaluate_metrics(targets, preds):
    return {
        'accuracy': float(accuracy_score(targets, preds)),
        'macro_f1': float(f1_score(targets, preds, average='macro')),
        'confusion_matrix': confusion_matrix(targets, preds),
        'classification_report': classification_report(
            targets, preds, target_names=[CLASS_INFO[i]['name'] for i in range(4)], zero_division=0
        )
    }

In [ ]:
set_seed(SEED)
images, labels = load_mat_dataset(MAT_PATH)
print('images shape:', images.shape)
print('labels shape:', labels.shape)
print('label range:', labels.min(), labels.max())

split = stratified_split(images, labels, SEED)
train_loader, val_loader, test_loader = build_loaders(split, BATCH_SIZE)

print('train:', split.train_images.shape[0])
print('val  :', split.val_images.shape[0])
print('test :', split.test_images.shape[0])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SmallCNN(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_val_f1 = -1.0
history = []

In [ ]:
for epoch in range(1, EPOCHS + 1):
    train_loss, train_targets, train_preds = run_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_targets, val_preds = run_epoch(model, val_loader, criterion, None, device)

    train_metrics = evaluate_metrics(train_targets, train_preds)
    val_metrics = evaluate_metrics(val_targets, val_preds)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_accuracy': train_metrics['accuracy'],
        'train_macro_f1': train_metrics['macro_f1'],
        'val_accuracy': val_metrics['accuracy'],
        'val_macro_f1': val_metrics['macro_f1'],
    })

    print(
        f'Epoch {epoch:03d}/{EPOCHS} | '
        f'train_loss={train_loss:.4f} val_loss={val_loss:.4f} | '
        f'train_f1={train_metrics['macro_f1']:.4f} val_f1={val_metrics['macro_f1']:.4f}'
    )

    if val_metrics['macro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['macro_f1']
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pt')

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pt', map_location=device))
test_loss, test_targets, test_preds = run_epoch(model, test_loader, criterion, None, device)
test_metrics = evaluate_metrics(test_targets, test_preds)

print('\n=== Final test metrics ===')
print('Accuracy :', round(test_metrics['accuracy'], 4))
print('Macro-F1 :', round(test_metrics['macro_f1'], 4))
print('Confusion matrix:\n', test_metrics['confusion_matrix'])
print('\nClassification report:\n')
print(test_metrics['classification_report'])

In [ ]:
summary = {
    'dataset': MAT_PATH,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'seed': SEED,
    'class_definition': CLASS_INFO,
    'split_sizes': {
        'train': int(split.train_labels.shape[0]),
        'val': int(split.val_labels.shape[0]),
        'test': int(split.test_labels.shape[0]),
    },
    'test_loss': float(test_loss),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_macro_f1': float(test_metrics['macro_f1']),
    'history': history,
}

with open(OUTPUT_DIR / 'training_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

np.save(OUTPUT_DIR / 'test_targets.npy', test_targets)
np.save(OUTPUT_DIR / 'test_predictions.npy', test_preds)

print('Saved outputs to:', OUTPUT_DIR)